# Bi-invariant Geodesic Regression

> **_Tip:_** Launch live version of this tutorial: [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/morphomatics/morphomatics.github.io/HEAD?filepath=docs%2Ftutorials%2Ftutorial_biinvariant_geodesic_regression.ipynb)

Many real-world phenomena are best described through continuous transformations, such as shape variations in medical imaging or movements in robotic systems. To effectively model the variability in such datasets, statistical analysis must be performed on **Lie groups**—mathematical structures that combine the properties of manifolds and groups. Since Lie groups inherently represent the symmetries present in the data, it is desirable to use statistical methods that adhere to these symmetries. (In technical terms: The result of the statical method should be equivariant or invariant under left and right translations of the data.) Doing so ensures that analyses are not influenced by arbitrary choices, such as that of a reference coordinate system.

Applying this idea to geodesic regression (see the tutorial on Riemannian regression), Morphomatics implements **bi-invariant geodesic regression**. For the definition, see **[Bi-invariant Geodesic Regression
with Data from the Osteoarthritis Initiative](https://arxiv.org/pdf/2502.11826v1)**.
In this tutorial, we demonstrate the difference between bi-invariant and Riemannian geodesic regression when applied to data from the group of rigid-body motions.

We start by creating random samples along a Riemannian geodesic.

In [2]:
%env JAX_PLATFORM_NAME=cpu

import jax
import jax.numpy as jnp
import jax.random as rnd
import numpy as np

from morphomatics.manifold import SE3
from morphomatics.stats import RiemannianRegression # BiinvariantRegression

import pyvista as pv

# G_affine = SE3(structure='AffineGroup')
G_riemannian = SE3(structure='CanonicalRiemannian')

# random endpoints of geodesic (could also use G_affine since both use the same rand-method)
p, q = jax.vmap(G_riemannian.rand)(rnd.split(rnd.key(3), 2))

# random samples along geodesic
N = 10
X = jnp.linspace(0, 1, N)
gam = jax.vmap(G_riemannian.connec.geopoint, (None, None, 0))(p, q, X)
noise = jax.vmap(G_riemannian.group.coords_inv)(rnd.normal(rnd.key(13), (N, G_riemannian.dim)))
Y = jax.vmap(G_riemannian.connec.exp)(gam, .1*noise)

env: JAX_PLATFORM_NAME=cpu


Now, we fit a Riemannian geodesic to the data points. It will approximate the geodesic that was used to sample the data.

In [6]:
regression_riemannian = RiemannianRegression
# regression_biinvariant = BiinvariantRegression

# Riemannian regression
beta = regression_riemannian(G_riemannian, Y, X).trend
f = G_riemannian.rand(rnd.key(1111111))
Y_f = jax.vmap(G_riemannian.group.righttrans, (0, None))(Y, f)
beta_f = regression_riemannian(G_riemannian, Y_f, X).trend


reg = regression_riemannian(G_riemannian, Y, X)
print(reg.unexplained_variance, reg.R2statistic)
for g in jax.vmap(G_riemannian.rand)(rnd.split(rnd.key(84), 100)):
    R2 = regression_riemannian(G_riemannian, jax.vmap(G_riemannian.group.righttrans, (0, None))(Y, g), X).R2statistic
    print(R2)

# show data

# glyph = pv.Text3D('T', height=.5, depth=.1)
#args = {'tip_radius': 0.2, 'shaft_radius': 0.1, 'scale': .5}
#glyph = pv.Arrow(**args)
glyph = pv.read('../data/paper-plane.obj').scale(.2).rotate_x(90).rotate_y(-45)

def frame(g: np.array):
    """Add a frame to the plotter."""
    # args = {'tip_radius': 0.02, 'shaft_radius': 0.01, 'scale': .5}
    # return pv.MultiBlock([
    #     pv.Arrow(start=g[:3, 3], direction=g[:3, 0], **args),
    #     pv.Arrow(start=g[:3, 3], direction=g[:3, 1], **args),
    #     pv.Arrow(start=g[:3, 3], direction=g[:3, 2], **args)
    # ])
    return glyph.copy().transform(g)

def tube_mesh(pts: np.array, interval=(0,1)):
    n = len(pts)-1
    curve = pv.PolyData(pts)
    # curve["time"] = np.linspace(*interval, curve.n_points)
    curve.lines = np.c_[np.full(n,2), np.arange(n), np.arange(n)+1]
    curve.tube(radius=0.01, inplace=True)
    return curve

def trace_frame(g: np.array):
    # scale = .5
    # return pv.MultiBlock([
    #     tube_mesh(g[:, :3, 3]),
    #     tube_mesh(g[:, :3, 3] + scale * g[:, :3, 0]),
    #     tube_mesh(g[:, :3, 3] + scale * g[:, :3, 1]),
    #     tube_mesh(g[:, :3, 3] + scale * g[:, :3, 2])
    # ])
    return pv.MultiBlock([glyph.copy().transform(f) for f in g])

plt = pv.Plotter()

# add samples
for i, g in enumerate(Y_f):
    plt.add_mesh(frame(np.asarray(g)), color='lime', inplace=True)

# add ground truth geodesic
# plt.add_mesh(tube_mesh(np.asarray(gam[:, :3, 3])), color='b')
# plt.add_mesh(trace_frame(np.asarray(gam)), color='b', opacity=1)

# add regression geodesic
geo_args = {'smooth_shading': True, 'opacity': 1}
t = jnp.linspace(0, 1, 200)
b_f = jax.vmap(G_riemannian.group.righttrans, (0, None))(jax.vmap(beta.eval)(t), f)
plt.add_mesh(trace_frame(np.asarray(b_f)), color='orange', opacity=.04)
b_f = jax.vmap(G_riemannian.group.righttrans, (0, None))(jax.vmap(beta.eval)(X), f)
plt.add_mesh(trace_frame(np.asarray(b_f)), color='orange', **geo_args)

plt.add_mesh(trace_frame(np.asarray(jax.vmap(beta_f.eval)(t))), color='r', opacity=.04, inplace=True)
plt.add_mesh(trace_frame(np.asarray(jax.vmap(beta_f.eval)(X))), color='r', inplace=True, **geo_args)

# show Riemannian geodesic between translated endpoints of regression one
plt.add_mesh(trace_frame(np.asarray(jax.vmap(G_riemannian.connec.geopoint, (None, None, 0))(b_f[0], b_f[-1], t))), color='yellow', opacity=.04, inplace=True)
plt.add_mesh(trace_frame(np.asarray(jax.vmap(G_riemannian.connec.geopoint, (None, None, 0))(b_f[0], b_f[-1], X))), color='yellow', inplace=True, **geo_args)

plt.show()

0.0877924 0.93148124
0.8579879
0.91832983
0.8985831
0.88533545
0.9283541
0.9146205
0.889802
0.913374
0.895677
0.9212794
0.92377806
0.9100581
0.91118175
0.93379027
0.9195922
0.9309917
0.90095574
0.9076886
0.8575464
0.8897834
0.8587941
0.86556304
0.89788973
0.89126384
0.862231
0.9078524
0.8547382
0.8888224
0.91491103
0.903654
0.88647133
0.9362489
0.9182641
0.91992295
0.88145727
0.83386075
0.89003265
0.88816625
0.9134571
0.8832738
0.8998824
0.8675035
0.89950454
0.92400634
0.8840364
0.92190987
0.87956303
0.93109715
0.8695316
0.9233189
0.87249494
0.8838298
0.8871475
0.9017173
0.8916482
0.9167503
0.90035194
0.9332314
0.8574093
0.914312
0.8980456
0.930225
0.91698664
0.8728013
0.8911539
0.8840013
0.884253
0.93521905
0.86554813
0.8783851
0.9177256
0.90415597
0.86440396
0.9210298
0.85237277
0.9015534
0.8531345
0.90947205
0.8837156
0.92169523
0.9095343
0.9227233
0.903347
0.8555066
0.87331927
0.8817211
0.8679743
0.9203018
0.91446817
0.90023977
0.8608309
0.9004045
0.915692
0.8900422
0.9356776
0.922

DeprecationError: The default value of `inplace` for the filter `PolyData.transform` will change in the future. Previously it defaulted to `True`, but will change to `False`. Explicitly set `inplace` to `True` or `False`.